# Import Libraries

In [28]:
import math
from pulp import LpProblem, LpVariable, LpStatus, lpSum, LpMaximize, LpInteger, LpContinuous, value
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
import pulp
import random
import warnings
from IPython.display import display
import pandas as pd
import numpy as np
import pulp as pl
from itertools import product

warnings.filterwarnings('ignore')

# Read Data Files

In [29]:
bom = pd.read_csv('data/bill_of_materials.csv')
cakes = pd.read_csv('data/cakes.csv')
channels = pd.read_csv('data/channels.csv')
ingredients = pd.read_csv('data/ingredients.csv')
demand_params = pd.read_csv('data/instructor_demand_competition.csv')
wages_energy = pd.read_csv('data/wages_energy.csv')
price_table = pd.read_csv('data/price_table_template.csv')

# Paramater Preperation:

In [30]:
ingredient_cost = ingredients.set_index('ingredient')['unit_cost_usd']
usage = bom.set_index('cake_id')[ingredients['ingredient'].tolist()].fillna(0)
cake_info = cakes.set_index('cake_id')
channel_info = channels.set_index('channel')
w_params = wages_energy.set_index('parameter')['value']
prep_wage_per_minute = float(w_params['prep_wage_usd_per_hour']) / 60
oven_wage_per_minute = float(w_params['oven_wage_usd_per_hour']) / 60
pack_wage_per_minute = float(w_params['pack_wage_usd_per_hour']) / 60
oven_rental_per_minute = float(w_params['oven_rental_usd_per_hour']) / 60
oven_cost_per_minute = float(w_params['oven_cost_usd_per_hour']) / 60
budget = float(w_params['budget_usd'])


cakes_list = cake_info.index.tolist()
channels_list = channel_info.index.tolist()
service_cap = channel_info['service_cap_per_week'] 
transport_cost = channel_info['transport_cost_per_unit_usd']
batch_size = cake_info['batch_size_units']
prep_time_per_unit = cake_info['prep_min_per_unit']
pack_time_per_unit = cake_info['pack_min_per_unit']
pack_cost_per_unit = cake_info['packaging_cost_per_unit_usd']
min_prod = cake_info['minimum_units_if_made']
oven_time_per_batch = cake_info['oven_min_per_batch']

cost_ing = (usage * ingredient_cost).sum(axis=1).to_dict()


wage_prep = prep_wage_per_minute * 60 
wage_pack = pack_wage_per_minute * 60
wage_decor = wage_prep 
oven_rental_per_min = oven_rental_per_minute
electricity_per_min = oven_cost_per_minute


price_pivot = price_table.pivot(index='cake', columns='channel', values='price').reindex(index=cakes_list, columns=channels_list)
price_pivot = price_pivot.fillna(19.0)


alpha_pivot = demand_params.pivot(index='ID', columns='channel', values='alpha').reindex(index=cakes_list, columns=channels_list)
beta_pivot = demand_params.pivot(index='ID', columns='channel', values='beta').reindex(index=cakes_list, columns=channels_list)

dem = (alpha_pivot - beta_pivot * price_pivot).clip(lower=0)

# IPL Model with set Prices

In [31]:
m = pl.LpProblem("Sweet_Market_Simple_ILP", pl.LpMaximize)
y = pl.LpVariable.dicts("y", (cakes_list, channels_list), lowBound=0, cat=pl.LpInteger)
s = pl.LpVariable.dicts("s", (cakes_list, channels_list), lowBound=0, cat=pl.LpInteger)
b = pl.LpVariable.dicts("b", cakes_list, lowBound=0, cat=pl.LpInteger)


for i in cakes_list:
    for j in channels_list:
        Dij = float(dem.loc[i, j])
        m += s[i][j] <= Dij
        m += s[i][j] <= y[i][j]


for j in channels_list:
    m += pl.lpSum(s[i][j] for i in cakes_list) <= float(service_cap.loc[j])


z = pl.LpVariable.dicts('z_make', cakes_list, lowBound=0, upBound=1, cat=pl.LpInteger)
M_big = {i: float(dem.loc[i].sum()) for i in cakes_list} 
for i in cakes_list:
    m += pl.lpSum(y[i][j] for j in channels_list) == int(batch_size.loc[i]) * b[i]
    if int(min_prod.loc[i]) > 0:
        m += pl.lpSum(y[i][j] for j in channels_list) - int(min_prod.loc[i]) * z[i] >= 0
        m += pl.lpSum(y[i][j] for j in channels_list) - M_big[i] * z[i] <= 0
    else:
        m += pl.lpSum(y[i][j] for j in channels_list) - M_big[i] * z[i] <= 0


revenue = pl.lpSum(float(price_pivot.loc[i,j]) * s[i][j] for i in cakes_list for j in channels_list)


c_ing = pl.lpSum(float(cost_ing.get(i, 0.0)) * y[i][j] for i in cakes_list for j in channels_list)
c_prep = pl.lpSum(float(prep_time_per_unit.loc[i]) * y[i][j] for i in cakes_list for j in channels_list) * (wage_prep / 60.0)


c_decor = 0 * pl.lpSum(y[i][j] for i in cakes_list for j in channels_list) * (wage_decor / 60.0)
c_pack_labor = pl.lpSum(float(pack_time_per_unit.loc[i]) * y[i][j] for i in cakes_list for j in channels_list) * (wage_pack / 60.0)
c_pack_mat = pl.lpSum(float(pack_cost_per_unit.loc[i]) * y[i][j] for i in cakes_list for j in channels_list)
total_oven_minutes = pl.lpSum(float(oven_time_per_batch.loc[i]) * b[i] for i in cakes_list)
c_oven = total_oven_minutes * (oven_rental_per_min + electricity_per_min)
c_transport = pl.lpSum(float(transport_cost.loc[j]) * s[i][j] for i in cakes_list for j in channels_list)
total_cost = c_ing + c_prep + c_decor + c_pack_labor + c_pack_mat + c_oven + c_transport


m += revenue - total_cost

m += total_cost <= budget


_ = m.solve(pl.PULP_CBC_CMD(msg=False))
status = pl.LpStatus[m.status]
profit = pl.value(m.objective)


print("Status:", status)
print("Profit: ${:,.2f}".format(profit))

rows = []
for i in cakes_list:
    for j in channels_list:
        rows.append({
            'cake': i,
            'channel': j,
            'price': float(price_pivot.loc[i, j]),
            'demand_cap': float(dem.loc[i, j]),
            'produced_y': int(pl.value(y[i][j])),
            'sold_s': int(pl.value(s[i][j]))
        })
df_plan = pd.DataFrame(rows)
df_plan.to_csv('results/plan.csv', index=False)
df_batches = pd.DataFrame({
    'cake': cakes_list,
    'batches': [int(pl.value(b[i])) for i in cakes_list]
})
df_batches.to_csv('results/batches.csv', index=False)
def v(x): return float(pl.value(x))
breakdown = {
    'Revenue': v(revenue),
    'Ingredients': v(c_ing),
    'Prep labor': v(c_prep),
    'Decor labor': v(c_decor),
    'Packaging labor': v(c_pack_labor),
    'Packaging material': v(c_pack_mat),
    'Oven (rental+energy)': v(c_oven),
    'Transportation': v(c_transport),
    'Total cost': v(total_cost),
    'Profit': v(revenue - total_cost)
}
pd.DataFrame(list(breakdown.items()), columns=['Item', 'Value ($)']).to_csv('results/costs.csv', index=False)
print('Saved: results/plan.csv, results/batches.csv, results/costs.csv')

Status: Optimal
Profit: $4,230.89
Saved: results/plan.csv, results/batches.csv, results/costs.csv


# Compute Prices Ranges

In [32]:

I = len(cakes_list)
J = len(channels_list)


cost_ingredients = [float(cost_ing[i]) for i in cakes_list]
labor_cost = [
    float(prep_time_per_unit.loc[i]) * (wage_prep / 60.0) +
    float(pack_time_per_unit.loc[i]) * (wage_pack / 60.0) for i in cakes_list]
utilities_cost = [
    (float(oven_time_per_batch.loc[i]) / float(batch_size.loc[i])) * (oven_rental_per_min + electricity_per_min) for i in cakes_list]
transportation_cost = [float(transport_cost.loc[j]) for j in channels_list]

alpha = [[float(alpha_pivot.loc[i, j]) for j in channels_list] for i in cakes_list]
beta = [[float(beta_pivot.loc[i, j]) for j in channels_list] for i in cakes_list]


P = [[None for _ in range(J)] for _ in range(I)]
eps = 1e-12
for i_idx, i in enumerate(cakes_list):
    for j_idx, j in enumerate(channels_list):
        p0 = cost_ingredients[i_idx] - labor_cost[i_idx] - utilities_cost[i_idx] - transportation_cost[j_idx]
        a = alpha[i_idx][j_idx]
        if a <= eps:
            raise ValueError(f"alpha[{i_idx}][{j_idx}] is zero or negative.")
        else:
            b = beta[i_idx][j_idx]
            if abs(b) <= eps:
                raise ValueError(f"beta[{i_idx}][{j_idx}] is zero or near zero.")
            p1 = a / b
        P[i_idx][j_idx] = [p0, p1]

        
for i_idx in range(I):
    for j_idx in range(J):
        print(f"P[{i_idx}][{j_idx}] = {P[i_idx][j_idx]}")

P[0][0] = [1.6692222222222226, 31.511453179402604]
P[0][1] = [1.4692222222222227, 16.184206095302237]
P[0][2] = [1.2692222222222225, 25.151562910428158]
P[1][0] = [1.4831666666666667, 33.053642485460664]
P[1][1] = [1.2831666666666668, 14.127162993238855]
P[1][2] = [1.0831666666666666, 24.54833663572844]
P[2][0] = [2.700666666666666, 31.805493523894597]
P[2][1] = [2.500666666666666, 16.01897803139296]
P[2][2] = [2.300666666666666, 21.3613496008435]
P[3][0] = [2.095833333333333, 34.24521354933726]
P[3][1] = [1.8958333333333335, 15.412570088806804]
P[3][2] = [1.6958333333333333, 22.753309324508468]
P[4][0] = [6.121, 29.87179487179487]
P[4][1] = [5.921, 13.68832902158313]
P[4][2] = [5.721000000000001, 20.34972292388729]
P[5][0] = [3.976, 33.571166095051446]
P[5][1] = [3.7760000000000002, 14.286200877080939]
P[5][2] = [3.576, 22.676841320760833]
P[6][0] = [0.6356666666666666, 35.21969080553295]
P[6][1] = [0.43566666666666654, 13.214095472239295]
P[6][2] = [0.23566666666666658, 22.7345174579

# Use Coarse to Fine Search to Maximise Prices in LP problem

In [33]:
import math
from functools import lru_cache
from typing import Callable, Sequence, Tuple

def coarse_to_fine_maximize(
    f: Callable[[float], float], lo: float, hi: float, steps: Sequence[float]
) -> Tuple[float, float]:
    if hi < lo:
        lo, hi = hi, lo
    x_best, f_best = lo, f(lo)
    for step in steps:
        if step <= 0:
            raise ValueError("Step sizes must be positive.")

        n = max(1, int(math.floor((hi - lo) / step)))
        xs = [lo + k * step for k in range(n + 1)]
        if xs[-1] < hi - 1e-12:
            xs.append(hi)

        vals = [f(x) for x in xs]
        k_best = max(range(len(xs)), key=lambda k: (vals[k], xs[k]))
        x_best, f_best = xs[k_best], vals[k_best]

        left  = xs[k_best - 1] if k_best - 1 >= 0 else xs[k_best]
        right = xs[k_best + 1] if k_best + 1 < len(xs) else xs[k_best]
        lo, hi = left, right
    return x_best, f_best

def _solve_profit_for_price_df(price_df: pd.DataFrame):
    dem_cur = (alpha_pivot - beta_pivot * price_df).clip(lower=0)

    m_loc = pl.LpProblem('ILP_c2f', pl.LpMaximize)
    y_loc = pl.LpVariable.dicts('y', (cakes_list, channels_list), 0, None, pl.LpInteger)
    s_loc = pl.LpVariable.dicts('s', (cakes_list, channels_list), 0, None, pl.LpInteger)
    b_loc = pl.LpVariable.dicts('b', cakes_list, 0, None, pl.LpInteger)
    z_loc = pl.LpVariable.dicts('z_make', cakes_list, 0, 1, pl.LpInteger)

    for i in cakes_list:
        for j in channels_list:
            Dij = float(dem_cur.loc[i, j])
            m_loc += s_loc[i][j] <= Dij
            m_loc += s_loc[i][j] <= y_loc[i][j]

    for j in channels_list:
        m_loc += pl.lpSum(s_loc[i][j] for i in cakes_list) <= float(service_cap.loc[j])

    M_big_cur = {i: float(dem_cur.loc[i].sum()) for i in cakes_list}
    for i in cakes_list:
        m_loc += pl.lpSum(y_loc[i][j] for j in channels_list) == int(batch_size.loc[i]) * b_loc[i]
        if int(min_prod.loc[i]) > 0:
            m_loc += pl.lpSum(y_loc[i][j] for j in channels_list) - int(min_prod.loc[i]) * z_loc[i] >= 0
            m_loc += pl.lpSum(y_loc[i][j] for j in channels_list) - M_big_cur[i] * z_loc[i] <= 0
        else:
            m_loc += pl.lpSum(y_loc[i][j] for j in channels_list) - M_big_cur[i] * z_loc[i] <= 0

    revenue_loc = pl.lpSum(float(price_df.loc[i, j]) * s_loc[i][j] for i in cakes_list for j in channels_list)
    c_ing_loc   = pl.lpSum(float(cost_ing.get(i, 0.0)) * y_loc[i][j] for i in cakes_list for j in channels_list)
    c_prep_loc  = pl.lpSum(float(prep_time_per_unit.loc[i]) * y_loc[i][j] for i in cakes_list for j in channels_list) * (wage_prep / 60.0)
    c_decor_loc = 0
    c_pack_lab_loc = pl.lpSum(float(pack_time_per_unit.loc[i]) * y_loc[i][j] for i in cakes_list for j in channels_list) * (wage_pack / 60.0)
    c_pack_mat_loc = pl.lpSum(float(pack_cost_per_unit.loc[i]) * y_loc[i][j] for i in cakes_list for j in channels_list)
    total_oven_minutes_loc = pl.lpSum(float(oven_time_per_batch.loc[i]) * b_loc[i] for i in cakes_list)
    c_oven_loc = total_oven_minutes_loc * (oven_rental_per_min + electricity_per_min)
    c_transport_loc = pl.lpSum(float(transport_cost.loc[j]) * s_loc[i][j] for i in cakes_list for j in channels_list)

    total_cost_loc = (c_ing_loc + c_prep_loc + c_decor_loc +
                      c_pack_lab_loc + c_pack_mat_loc + c_oven_loc + c_transport_loc)

    m_loc += total_cost_loc <= budget
    m_loc += revenue_loc - total_cost_loc

    m_loc.solve(pl.PULP_CBC_CMD(msg=False))
    return pl.LpStatus[m_loc.status], pl.value(revenue_loc - total_cost_loc)



def make_profit_fn_for_pair(i_val, j_val, base_price_df: pd.DataFrame) -> Callable[[float], float]:
    @lru_cache(maxsize=None)
    def _f_cents(x_cents: int) -> float:
        x = x_cents / 100.0
        price_df = base_price_df.copy()
        price_df.loc[i_val, j_val] = x
        status, profit = _solve_profit_for_price_df(price_df)
        return -1e18 if status != 'Optimal' else float(profit)

    return lambda x: _f_cents(int(round(x * 100)))

def compute_price_bounds(i_val, j_val, cur_price: float) -> Tuple[float, float]:

    a = float(alpha_pivot.loc[i_val, j_val])
    b = float(beta_pivot.loc[i_val, j_val])
    lo = 0.0
    if b > 0:
        hi = max(cur_price, a / b)
    else:
        hi = max(cur_price * 2.0, cur_price + 1.0)  
    if hi <= lo:
        hi = lo + 0.01
    return lo, hi

steps = [1.0, 0.1, 0.01]
best_prices_df = price_pivot.copy()

for i_val in cakes_list:
    for j_val in channels_list:
        cur = float(best_prices_df.loc[i_val, j_val])
        p_lo, p_hi = compute_price_bounds(i_val, j_val, cur)
        profit_fn = make_profit_fn_for_pair(i_val, j_val, best_prices_df)
        x_star, _ = coarse_to_fine_maximize(profit_fn, p_lo, p_hi, steps)
        best_prices_df.loc[i_val, j_val] = x_star 


best_prices_df.to_csv('results/best_prices_corse_to_fine.csv')
print('Saved results/best_prices_corse_to_fine.csv')

price_pivot = best_prices_df.copy()

dem = (alpha_pivot - beta_pivot * price_pivot).clip(lower=0)


Saved results/best_prices_corse_to_fine.csv


# Rerun ILP With Optimal Prices Found in Coarse to Fine

In [34]:
m = pl.LpProblem("Sweet_Market_Simple_ILP", pl.LpMaximize)


y = pl.LpVariable.dicts("y", (cakes_list, channels_list), lowBound=0, cat=pl.LpInteger)
s = pl.LpVariable.dicts("s", (cakes_list, channels_list), lowBound=0, cat=pl.LpInteger)
b = pl.LpVariable.dicts("b", cakes_list, lowBound=0, cat=pl.LpInteger)

for i in cakes_list:
    for j in channels_list:
        Dij = float(dem.loc[i, j])
        m += s[i][j] <= Dij
        m += s[i][j] <= y[i][j]

for j in channels_list:
    m += pl.lpSum(s[i][j] for i in cakes_list) <= float(service_cap.loc[j])

z = pl.LpVariable.dicts('z_make', cakes_list, lowBound=0, upBound=1, cat=pl.LpInteger)
M_big = {i: float(dem.loc[i].sum()) for i in cakes_list}
for i in cakes_list:
    m += pl.lpSum(y[i][j] for j in channels_list) == int(batch_size.loc[i]) * b[i]
    if int(min_prod.loc[i]) > 0:
        m += pl.lpSum(y[i][j] for j in channels_list) - int(min_prod.loc[i]) * z[i] >= 0
        m += pl.lpSum(y[i][j] for j in channels_list) - M_big[i] * z[i] <= 0
    else:
        m += pl.lpSum(y[i][j] for j in channels_list) - M_big[i] * z[i] <= 0

revenue = pl.lpSum(float(price_pivot.loc[i, j]) * s[i][j] for i in cakes_list for j in channels_list)

c_ing = pl.lpSum(float(cost_ing.get(i, 0.0)) * y[i][j] for i in cakes_list for j in channels_list)
c_prep = pl.lpSum(float(prep_time_per_unit.loc[i]) * y[i][j] for i in cakes_list for j in channels_list) * (wage_prep / 60.0)
c_decor = 0 * pl.lpSum(y[i][j] for i in cakes_list for j in channels_list) * (wage_decor / 60.0)
c_pack_labor = pl.lpSum(float(pack_time_per_unit.loc[i]) * y[i][j] for i in cakes_list for j in channels_list) * (wage_pack / 60.0)
c_pack_mat = pl.lpSum(float(pack_cost_per_unit.loc[i]) * y[i][j] for i in cakes_list for j in channels_list)
total_oven_minutes = pl.lpSum(float(oven_time_per_batch.loc[i]) * b[i] for i in cakes_list)
c_oven = total_oven_minutes * (oven_rental_per_min + electricity_per_min)
c_transport = pl.lpSum(float(transport_cost.loc[j]) * s[i][j] for i in cakes_list for j in channels_list)
total_cost = c_ing + c_prep + c_decor + c_pack_labor + c_pack_mat + c_oven + c_transport

m += revenue - total_cost
m += total_cost <= budget

_ = m.solve(pl.PULP_CBC_CMD(msg=False))
status = pl.LpStatus[m.status]
profit = pl.value(m.objective)
print("Status:", status)
print("Profit: ${:,.2f}".format(profit))

rows = []
for i in cakes_list:
    for j in channels_list:
        rows.append({
            'cake': i,
            'channel': j,
            'price': float(price_pivot.loc[i, j]),
            'demand_cap': float(dem.loc[i, j]),
            'produced_y': int(pl.value(y[i][j])),
            'sold_s': int(pl.value(s[i][j]))
        })
df_plan = pd.DataFrame(rows)
df_plan.to_csv('results/plan_prices_coarse_to_fine.csv', index=False)

df_batches = pd.DataFrame({
    'cake': cakes_list,
    'batches': [int(pl.value(b[i])) for i in cakes_list]
})
df_batches.to_csv('results/batches_prices_coarse_to_fine.csv', index=False)

def v(x): return float(pl.value(x))
breakdown = {
    'Revenue': v(revenue),
    'Ingredients': v(c_ing),
    'Prep labor': v(c_prep),
    'Decor labor': v(c_decor),
    'Packaging labor': v(c_pack_labor),
    'Packaging material': v(c_pack_mat),
    'Oven (rental+energy)': v(c_oven),
    'Transportation': v(c_transport),
    'Total cost': v(total_cost),
    'Profit': v(revenue - total_cost)
}

pd.DataFrame(list(breakdown.items()), columns=['Item', 'Value ($)']).to_csv('results/costs_prices_coarse_to_fine.csv', index=False)
print('Saved: results/plan_prices_coarse_to_fine.csv, results/batches_prices_coarse_to_fine.csv, results/costs_prices_coarse_to_fine.csv')

Status: Optimal
Profit: $5,734.23
Saved: results/plan_prices_coarse_to_fine.csv, results/batches_prices_coarse_to_fine.csv, results/costs_prices_coarse_to_fine.csv
